<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_5/bonus/linux_devops/note_bonus_linux_devops_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐧 Бонус. Linux для розробника

| Крок | Що робимо |
|---|---|
| 0 | робоча папка й помічник `sh()` |
| 1 | пайплайни: журнал доступу → топ IP (вправа 1) |
| 2 | коди виходу, `&&`, `set -e`, `pipefail` (вправа 2) |
| 3 | права й користувачі (вправа 3) |
| 4 | процеси й сигнали (вправи 4–5) |
| 5 | порти: `127.0.0.1` чи `0.0.0.0` (вправа 6) |
| 6 | змінні середовища й `.env` (вправа 7) |
| 7 | скрипт, що падає вчасно (вправа 8) |
| 8 | знайди помилку: порожній бекап (вправа 9) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам.

Colab — це Linux (Ubuntu), тож усі команди справжні. Відповіді пишемо рядком bash у Python-змінній, а помічник `sh()` виконує його і повертає вивід і код виходу, щоб `assert` міг перевірити результат. Теорія — у книзі курсу: [Бонус. Linux](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/bonus_linux/).

---
## 0. Робоча папка

In [ ]:
import os
import random
import subprocess
from pathlib import Path

WORK = Path("linux_lab").resolve()
WORK.mkdir(exist_ok=True)
os.chdir(WORK)


def sh(script: str, check: bool = False, **kwargs) -> subprocess.CompletedProcess[str]:
    """Виконати рядок у bash у папці linux_lab; stdout і stderr — у результаті."""
    result = subprocess.run(["bash", "-c", script], capture_output=True, text=True, **kwargs)
    if check and result.returncode != 0:
        raise RuntimeError(f"код {result.returncode}: {result.stderr}")
    return result


print(sh("uname -srm; whoami; pwd; bash --version | head -1").stdout)

---
## 1. Пайплайни

Команда Linux робить одну річ і пише результат у **stdout**. `|` передає stdout однієї команди на stdin наступної. Так із маленьких програм збирають відповідь на питання, для якого окремої програми немає.

Згенеруємо журнал доступу nginx (такий пише сервер у `/var/log/nginx/access.log`, урок 50):

In [ ]:
random.seed(48)
ips = ["10.0.0.5"] * 30 + ["172.18.0.4"] * 22 + ["192.168.1.20"] * 9 + [f"203.0.113.{i}" for i in range(1, 40)]
paths = ["/api/news", "/api/news/stats", "/health", "/api/admin/token", "/docs"]
lines = []
for number in range(len(ips)):
    ip = random.choice(ips)
    status = random.choice([200] * 8 + [401, 429])
    lines.append(f'{ip} - - [28/Sep/2026:10:{number // 60:02}:{number % 60:02} +0000] "GET {random.choice(paths)} HTTP/1.1" {status} 512')
Path("access.log").write_text("\n".join(lines) + "\n")
print(sh("wc -l access.log; head -3 access.log").stdout)

### Вправа 1

Напиши пайплайн `top_ips`, що друкує **3** IP з найбільшою кількістю запитів у форматі `кількість IP` (як `uniq -c`), від найбільшого. Підказка: `cut -d' ' -f1`, `sort`, `uniq -c`, `sort -rn`, `head`.

In [ ]:
# YOUR CODE HERE

result = sh(top_ips)
print(result.stdout)

from collections import Counter
expected = Counter(line.split()[0] for line in lines).most_common(3)
got = [(parts[1], int(parts[0])) for parts in (line.split() for line in result.stdout.splitlines())]
assert got == expected, (got, expected)
print("✅ Вправа 1 пройдена")

Той самий пайплайн з `grep ' 429 '` на початку відповість, **хто** впирається в rate limit (урок 40). Жодного Python — і відповідь за секунду навіть для гігабайтного журналу.

---
## 2. Коди виходу

Кожна програма завершується **кодом виходу**: `0` — успіх, інше — помилка. `$?` — код останньої команди. `a && b` — `b` лише якщо `a` вдалась; `a || b` — `b` лише якщо `a` впала. На кодах виходу тримається все: CI вважає крок проваленим, якщо код не 0 (урок 51).

🔮 **Прогноз:** який код виходу у пайплайна `false | true`? А з `set -o pipefail`?

In [ ]:
for script in ("true", "false", "false | true", "set -o pipefail; false | true", "grep -q 'HTTP/1.1\" 500' access.log"):
    print(f"{script:<36} → код {sh(script).returncode}")

### Вправа 2

У скрипті три команди. Друга падає. Допиши на початок `script` **одну** опцію bash, щоб скрипт зупинився на помилці, і третій рядок не виконався.

In [ ]:
BODY = """
echo "крок 1: міграції"
ls /немає-такої-папки
echo "крок 3: запуск сервера"
"""
# YOUR CODE HERE

result = sh(script)
print(result.stdout, result.stderr, "код:", result.returncode)
assert "крок 1" in result.stdout and "крок 3" not in result.stdout and result.returncode != 0
print("✅ Вправа 2 пройдена")

---
## 3. Права й користувачі

`ls -l` показує власника, групу і права: `rwx` для власника, групи й решти. Цифрами: `r=4, w=2, x=1`, тож `640` = `rw-` власнику, `r--` групі, нічого решті. Контейнер уроку 49 працює від користувача `app` (UID 10001): код належить root і лише для читання, а в `/data` писати можна.

У Colab ти **root** — йому дозволено все. Щоб побачити права в дії, створимо звичайного користувача `student` і виконаємо команду від його імені (`runuser -u student -- …`).

In [ ]:
print(sh("id -u student >/dev/null 2>&1 || useradd --uid 10001 --no-create-home student; id student").stdout)
Path("app").mkdir(exist_ok=True)
Path("app/api.py").write_text("print('код застосунку')\n")
Path("data").mkdir(exist_ok=True)
Path(".env").write_text("JWT_SECRET=local-secret\n")
os.chmod(WORK, 0o755)
print(sh("ls -la").stdout)

### Вправа 3

Налаштуй права так, як в образі уроку 49:

- `app/api.py` — власник root, **усі можуть читати**, ніхто, крім root, не може змінити (`644`);
- папка `data` — власник `student` (`chown`), щоб застосунок міг туди писати;
- `.env` — лише власник читає й пише (`600`): `student` його не прочитає.

Запиши команди в `setup`.

In [ ]:
# YOUR CODE HERE

sh(setup, check=True)
as_student = lambda command: sh(f"runuser -u student -- bash -c '{command}'")
print(sh("ls -la app data .env; stat -c '%a %U %n' app/api.py data .env").stdout)
assert as_student("cat app/api.py").returncode == 0            # читати код можна
assert as_student("echo x >> app/api.py").returncode != 0      # змінювати — ні
assert as_student("touch data/news_hub.db").returncode == 0    # писати дані — так
assert as_student("cat .env").returncode != 0                  # секрет — ні
print("✅ Вправа 3 пройдена")

---
## 4. Процеси й сигнали

Запущена програма — **процес** з номером (PID). `ps` показує процеси, `kill -СИГНАЛ PID` надсилає сигнал:

| Сигнал | Номер | Що означає | Можна перехопити? |
|---|---|---|---|
| `SIGINT` | 2 | Ctrl+C | так |
| `SIGTERM` | 15 | «завершись, будь ласка» — `kill`, `systemctl stop`, `docker stop` | так |
| `SIGKILL` | 9 | «помри негайно» | ні |

Процес, убитий сигналом N, отримує код виходу `128 + N`: `143` після `SIGTERM`, `137` після `SIGKILL` (урок 49).

In [ ]:
server = subprocess.Popen(["sleep", "300"])
print("PID:", server.pid)
print(sh(f"ps -o pid,ppid,stat,cmd -p {server.pid}").stdout)

### Вправа 4

Знайди PID процесу `sleep 300` командою `pgrep` (`pgrep -n -x sleep` — найновіший процес з точною назвою `sleep`) і надішли йому `SIGTERM` командою `kill`. Запиши bash-команду в `stop`. Потім дізнайся код виходу процесу через `server.wait()` і збережи в `code`.

In [ ]:
# YOUR CODE HERE

print("код виходу:", code, "(у shell — 128 + 15 =", 128 + 15, ")")
assert code == -15                 # Python показує «вбитий сигналом 15» як -15
print("✅ Вправа 4 пройдена")

### Вправа 5. Прибрати за собою на `SIGTERM`

Сервер перед зупинкою має закрити з'єднання, дописати журнал, видалити тимчасові файли. У bash для цього — `trap 'команди' TERM`. Допиши `worker.sh`: на `SIGTERM` скрипт має записати `cleanup done` у `worker.log`, видалити `worker.lock` і завершитись з кодом `0`.

In [ ]:
WORKER_BODY = """
echo started > worker.log
touch worker.lock
while true; do sleep 0.1; done
"""
# YOUR CODE HERE
Path("worker.sh").write_text("#!/usr/bin/env bash\n" + TRAP + WORKER_BODY)

import time
worker = subprocess.Popen(["bash", "worker.sh"])
while not Path("worker.lock").exists():
    time.sleep(0.05)
worker.send_signal(15)
code = worker.wait(timeout=5)
print("код:", code, "| журнал:", Path("worker.log").read_text().split(), "| lock:", Path("worker.lock").exists())
assert code == 0 and "cleanup" in Path("worker.log").read_text() and not Path("worker.lock").exists()
print("✅ Вправа 5 пройдена")

<details>
<summary>А якщо скрипт запустити через <code>sh -c</code>?</summary>

Сигнал отримає оболонка `sh`, а не `worker.sh` — і прибирання не відбудеться. Саме тому в `Dockerfile` пишуть exec-форму `CMD ["uvicorn", …]`: застосунок стає першим процесом контейнера і сам отримує `SIGTERM` від `docker stop` (урок 49).
</details>

---
## 5. Порти: `127.0.0.1` чи `0.0.0.0`

Сервер слухає **адресу й порт**. `127.0.0.1` (loopback) — з'єднання лише з тієї ж машини; `0.0.0.0` — з усіх мережевих інтерфейсів. У контейнері «та сама машина» — сам контейнер, тому uvicorn у Docker запускають з `--host 0.0.0.0`.

Дізнаємось адресу цієї машини в мережі (не loopback):

In [ ]:
import socket

external_ip = sh("hostname -I").stdout.split()[0]
print("адреса в мережі:", external_ip)


def reachable(host: str, port: int) -> bool:
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

### Вправа 6

Запусти `python3 -m http.server` двічі у фоні (`subprocess.Popen`): на порту `8765` з `--bind 127.0.0.1` (`local`) і на `8766` з `--bind 0.0.0.0` (`everywhere`). Потім заповни словник `results`: чи доступний кожен сервер за адресою `127.0.0.1` і за `external_ip`.

🔮 **Прогноз:** який із чотирьох варіантів дасть `False`?

In [ ]:
# YOUR CODE HERE

for (port, host), ok in results.items():
    print(f"порт {port}, з'єднання на {host:<15} → {ok}")
local.terminate(); everywhere.terminate()
assert results == {(8765, "127.0.0.1"): True, (8765, external_ip): False,
                   (8766, "127.0.0.1"): True, (8766, external_ip): True}
print("✅ Вправа 6 пройдена")

---
## 6. Змінні середовища

Процес успадковує змінні середовища від батька. Звичайна змінна shell (`NAME=value`) дочірнім процесам **не** передається; `export NAME=value` — передається. Так `news_hub` отримує `DATABASE_URL`, `REDIS_URL`, секрети — і так їх передає Docker (`-e`, `--env-file`).

🔮 **Прогноз:** що надрукує Python у кожному рядку?

In [ ]:
for script in ('SECRET=abc; python3 -c "import os; print(os.getenv(\'SECRET\'))"',
               'export SECRET=abc; python3 -c "import os; print(os.getenv(\'SECRET\'))"',
               'SECRET=abc python3 -c "import os; print(os.getenv(\'SECRET\'))"'):
    print(f"{script[:22]:<24} → {sh(script).stdout.strip()}")

### Вправа 7. `$` у значенні

bcrypt-хеш пароля адміна (урок 47) містить `$`: `$2b$12$abc…`. У bash `$2` у подвійних лапках — це «другий аргумент скрипта» (порожній), тож хеш зіпсується. Файл `.env.admin` нижче завантажують так: `set -a; source .env.admin; set +a` (`set -a` — експортувати все, що оголошено).

Виправ **лише лапки** в `ENV_FILE`, щоб Python побачив хеш без змін.

In [ ]:
HASH = "$2b$12$eBDJbnUjhmlfW0bqlm9n1u"
# YOUR CODE HERE
Path(".env.admin").write_text(ENV_FILE)

broken = sh(f'ADMIN_PASSWORD_HASH="{HASH}"; echo "$ADMIN_PASSWORD_HASH"').stdout.strip()
loaded = sh('set -a; source .env.admin; set +a; python3 -c "import os; print(os.environ[\'ADMIN_PASSWORD_HASH\'])"').stdout.strip()
print("у подвійних лапках:", broken)
print("з .env.admin:      ", loaded)
assert loaded == HASH
print("✅ Вправа 7 пройдена")

Одинарні лапки в bash — «усе буквально». Тому `python -m news_hub.security` друкує хеш в одинарних лапках. Але `docker run --env-file` лапок не знімає зовсім — для нього окремий режим `--env-file` (урок 49).

---
## 7. Скрипт, що падає вчасно

Скрипт деплою чи бекапу запускають без людини — з cron, CI, systemd. Якщо він «продовжить після помилки», наслідки побачать пізно. Тому перший рядок таких скриптів:

```bash
set -euo pipefail
```

| Опція | Що робить |
|---|---|
| `-e` | зупинитись, якщо команда завершилась з помилкою |
| `-u` | помилка, якщо змінну не задано (замість порожнього рядка) |
| `-o pipefail` | пайплайн падає, якщо впала **будь-яка** його команда, а не лише остання |

### Вправа 8

Допиши `deploy.sh`: скрипт бере версію з першого аргументу (`$1`); без аргументу — друкує `usage: deploy.sh VERSION` у **stderr** і завершується з кодом `2`. З аргументом — друкує `deploying VERSION`.

In [ ]:
# YOUR CODE HERE
Path("deploy.sh").write_text(DEPLOY)
sh("chmod +x deploy.sh", check=True)

no_args, with_arg = sh("./deploy.sh"), sh("./deploy.sh v48")
print(repr(no_args.stderr), no_args.returncode, "|", repr(with_arg.stdout), with_arg.returncode)
assert no_args.returncode == 2 and "usage" in no_args.stderr and no_args.stdout == ""
assert with_arg.returncode == 0 and with_arg.stdout.strip() == "deploying v48"
print("✅ Вправа 8 пройдена")

---
## 8. Знайди помилку: порожній бекап

Скрипт бекапу бази щоночі: `pg_dump` → `gzip` → файл. Одного разу пароль до бази змінився. Скрипт відпрацював з кодом 0, файл бекапу з'явився — і ніхто нічого не помітив. `pg_dump` тут замінено функцією, яка падає, як `pg_dump` з неправильним паролем.

In [ ]:
BACKUP = """#!/usr/bin/env bash
set -e
pg_dump() { echo "pg_dump: error: connection failed: password authentication failed" >&2; return 1; }
pg_dump news_hub | gzip > backup.sql.gz
echo "бекап готовий: $(stat -c %s backup.sql.gz) байт"
"""
Path("backup.sh").write_text(BACKUP)
result = sh("bash backup.sh")
print(result.stdout, result.stderr, "код:", result.returncode)

### Вправа 9

Чому `set -e` не зупинив скрипт? Виправ `BACKUP` одним словом у рядку `set -e`, щоб скрипт завершився з помилкою і не друкував «бекап готовий».

In [ ]:
# YOUR CODE HERE
Path("backup.sh").write_text(FIXED)
result = sh("bash backup.sh")
print(result.stdout, result.stderr, "код:", result.returncode)
assert result.returncode != 0 and "бекап готовий" not in result.stdout
print("✅ Вправа 9 пройдена")

<details>
<summary>Відповідь</summary>

Код виходу пайплайна — код **останньої** команди. `gzip` успішно стиснув порожній вхід (20 байт заголовка gzip), тож `pg_dump … | gzip` «вдався», і `set -e` нічого не побачив. `set -o pipefail` робить пайплайн проваленим, якщо впала будь-яка команда. Цей самий скрипт з'явиться в уроці 50 — для бекапу PostgreSQL з контейнера.
</details>

In [ ]:
os.chdir(WORK.parent)
sh(f"rm -rf {WORK}")
print("готово")

---
## Самоперевірка

1. Що передає `|` від однієї команди до іншої?
2. Який код виходу має процес, убитий `SIGKILL`, і чому його не можна перехопити?
3. Чому сервер у контейнері слухає `0.0.0.0`?
4. Чим `NAME=value` відрізняється від `export NAME=value`?
5. Навіщо `pipefail`, якщо вже є `set -e`?

## Далі

- Книга курсу: [Бонус. Linux](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/bonus_linux/) — ментальна модель, процеси й сигнали, права, мережа; довідник з 18 розділів.
- Урок 49 — Docker: контейнер — це процес Linux з власним користувачем, мережею й файловою системою.